# 02_03 - Limpieza de presión estructural SER

Este notebook prepara las fuentes complementarias del bloque SER que aproximan presión estructural o presión no observada directamente en los tiques pagados.

Fuentes tratadas:

1. `ser_autorizaciones`
2. `ser_padron_vehiculos_ivtm_barrio`

El objetivo es limpiar ambas fuentes de forma individual y dejar salidas `interim` reproducibles.

## 0. Objetivo y alcance

**Objetivo.** Limpiar y preparar fuentes complementarias SER que aproximan presión estructural/no observada:

- autorizaciones SER activas;
- turismos con distintivo 0 por barrio a partir del padrón IVTM.

**Uso en el TFM.**

- `ser_autorizaciones` aproxima presión potencial asociada a residentes y comerciales con autorización SER.
- `ser_padron_vehiculos_ivtm_barrio` aproxima stock anual de turismos Cero Emisiones por barrio, tras verificar que la etiqueta ambiental 0 también incluye otros tipos de vehículo no comparables con la presión SER ordinaria.

## 1. Referencia documental

La documentación oficial se usa como referencia para interpretar columnas, valores esperados y cambios de esquema, pero no sustituye la inspección real de los archivos raw. Los PDFS de referencia se encuentran en `docs/source_docs/ser/ser_autorizaciones` y `docs/source_docs/ser/ser_padron_vehiculos_ivtm_barrio`.


**Autorizaciones SER.** El documento oficial describe autorizaciones SER mensuales y anuales de residentes y comerciales activas en algún momento del periodo. También documenta columnas como `tipo_autorizacion`, `subtipo_autorizacion`, `cod_distrito`, `cod_distrito_barrio`, `estado`, `fecha_inicio_periodo`, `fecha_activacion`, `fecha_vigencia`, `distintivo`, `periodicidad`, `tipologia_vehiculo`, `tipologia_propulsion` y `clasificacion_industria`. El propio documento indica que, desde 2025, `fecha_inicio_periodo` diferencia el inicio real de validez frente a `fecha_activacion`.

**Padrón IVTM.** El documento oficial describe el padrón anual de vehículos por ejercicio, distrito, barrio, tipo de vehículo, etiqueta medioambiental, clasificación ambiental, carburante, año de matriculación y `CONTADOR`. En particular, `ETIQUETA_MEDIOAMBIENTAL = 0` corresponde a vehículos Cero Emisiones, por lo que debe normalizarse con cuidado para no confundir `0` con nulo o con valor numérico mal interpretado.


## 2. Configuración inicial

Se cargan dependencias, rutas del repositorio, catálogo y funciones auxiliares. Las rutas se derivan del repositorio y no deben depender de rutas absolutas del ordenador.


In [113]:
from pathlib import Path
import json
import re
import unicodedata
import warnings

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)

TARGET_DATASET_IDS = [
    "ser_autorizaciones",
    "ser_padron_vehiculos_ivtm_barrio",
]

WINDOW_START = pd.Timestamp("2023-01-01")
WINDOW_END = pd.Timestamp("2026-12-31")
EXPECTED_YEARS_IVTM = {2023, 2024, 2025}


def find_repo_root(start: Path | None = None) -> Path:
    start = Path.cwd() if start is None else Path(start)
    for candidate in [start, *start.parents]:
        if (candidate / "data_catalog.csv").exists():
            return candidate
    raise FileNotFoundError("No se ha encontrado data_catalog.csv en el directorio actual ni en sus padres.")


ROOT = find_repo_root()
CATALOG_PATH = ROOT / "data_catalog.csv"
SOURCE_DOCS_ROOT = ROOT / "docs" / "source_docs"

print("ROOT =", ROOT)
print("CATALOG_PATH exists =", CATALOG_PATH.exists())
print("SOURCE_DOCS_ROOT exists =", SOURCE_DOCS_ROOT.exists())

ROOT = /Users/hugo/TFM_parking_madrid
CATALOG_PATH exists = True
SOURCE_DOCS_ROOT exists = True


### 2.1. Funciones auxiliares

Estas funciones fijan criterios comunes de lectura, normalización y validación. No agregan información, no crean proxy y no construyen joins finales.


In [114]:
def normalize_text(value):
    if pd.isna(value):
        return pd.NA
    text = str(value).strip()
    text = re.sub(r"\s+", " ", text)
    return text


def normalize_key(value):
    if pd.isna(value):
        return pd.NA
    text = str(value).strip().lower()
    text = unicodedata.normalize("NFKD", text)
    text = "".join(ch for ch in text if not unicodedata.combining(ch))
    text = re.sub(r"[^a-z0-9]+", "_", text)
    text = re.sub(r"_+", "_", text).strip("_")
    return text


def normalize_columns(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out.columns = [normalize_key(c) for c in out.columns]
    return out


def to_numeric_series(series: pd.Series) -> pd.Series:
    # Conversión conservadora: respeta coma decimal si aparece y evita transformar texto no numérico sin control.
    s = series.astype("string").str.strip()
    s = s.str.replace(".", "", regex=False).where(s.str.contains(",", na=False), s)
    s = s.str.replace(",", ".", regex=False)
    return pd.to_numeric(s, errors="coerce")


def parse_date_series(series: pd.Series) -> pd.Series:
    return pd.to_datetime(series, errors="coerce", dayfirst=False)


def extract_year_from_filename(path) -> int | None:
    match = re.search(r"(20\d{2})", Path(path).name)
    return int(match.group(1)) if match else None


def read_csv_robust(path: Path, **kwargs) -> pd.DataFrame:
    encodings = ["utf-8", "utf-8-sig", "latin1"]
    separators = [";", ",", "\t"]
    last_error = None

    for encoding in encodings:
        for sep in separators:
            try:
                df = pd.read_csv(path, sep=sep, encoding=encoding, low_memory=False, **kwargs)
                if df.shape[1] > 1:
                    return df
            except Exception as exc:
                last_error = exc

    raise RuntimeError(f"No se pudo leer {path}. Último error: {last_error}")


def resolve_raw_files(raw_pattern: str) -> list[Path]:
    path = ROOT / raw_pattern
    if any(ch in raw_pattern for ch in ["*", "?", "["]):
        return sorted(ROOT.glob(raw_pattern))
    return [path] if path.exists() else []


def format_check_rows(rows: list[tuple[str, object, str]]) -> pd.DataFrame:
    return pd.DataFrame(rows, columns=["check", "valor", "interpretacion"])

print("Funciones auxiliares cargadas")

Funciones auxiliares cargadas


## 3. Catálogo y archivos raw

Se carga `data_catalog.csv`, se filtran las fuentes objetivo y se comprueba que los archivos raw existen antes de limpiar. Si falta alguna fuente o archivo, el notebook debe fallar pronto.


In [115]:
catalog = pd.read_csv(CATALOG_PATH)

missing_catalog = sorted(set(TARGET_DATASET_IDS) - set(catalog["dataset_id"]))
if missing_catalog:
    raise ValueError(f"Faltan dataset_id en data_catalog.csv: {missing_catalog}")

target_catalog = catalog[catalog["dataset_id"].isin(TARGET_DATASET_IDS)].copy()

catalog_rows = []
RAW_FILES = {}

for _, row in target_catalog.iterrows():
    dataset_id = row["dataset_id"]
    raw_files = resolve_raw_files(row["archivo_raw"])
    RAW_FILES[dataset_id] = raw_files

    catalog_rows.append({
        "dataset_id": dataset_id,
        "formato_preferido": row.get("formato_preferido"),
        "archivo_raw": row.get("archivo_raw"),
        "n_archivos_encontrados": len(raw_files),
        "archivos_encontrados": [str(p.relative_to(ROOT)) for p in raw_files],
        "archivo_interim": row.get("archivo_interim"),
    })

catalog_check = pd.DataFrame(catalog_rows)
display(catalog_check)

missing_raw = catalog_check[catalog_check["n_archivos_encontrados"].eq(0)]["dataset_id"].tolist()
if missing_raw:
    raise FileNotFoundError(f"Faltan archivos raw para: {missing_raw}")

,dataset_id,formato_preferido,archivo_raw,n_archivos_encontrados,archivos_encontrados,archivo_interim
0,ser_autorizaciones,csv,data/raw/ser/ser_autorizaciones/ser_autorizaci...,4,[data/raw/ser/ser_autorizaciones/ser_autorizac...,data/interim/ser/ser_autorizaciones/ser_autori...
1,ser_padron_vehiculos_ivtm_barrio,csv,data/raw/ser/ser_padron_vehiculos_ivtm_barrio/...,3,[data/raw/ser/ser_padron_vehiculos_ivtm_barrio...,data/interim/ser/ser_padron_vehiculos_ivtm_bar...


## 4. Inspección estructural global

La inspección global solo verifica lectura, forma y columnas normalizadas. La decisión de variables y validaciones específicas se realiza dentro de cada fuente.


In [116]:
RAW_TABLES = {}
inspection_rows = []

for dataset_id, files in RAW_FILES.items():
    parts = []
    for path in files:
        df_raw = read_csv_robust(path)
        df_raw = normalize_columns(df_raw)
        df_raw["archivo_origen"] = str(path.relative_to(ROOT))
        df_raw["anio_archivo"] = extract_year_from_filename(path)
        parts.append(df_raw)

    df_all = pd.concat(parts, ignore_index=True)
    RAW_TABLES[dataset_id] = df_all

    inspection_rows.append({
        "dataset_id": dataset_id,
        "n_archivos": len(files),
        "shape": df_all.shape,
        "columnas_normalizadas": list(df_all.columns),
    })

global_inspection = pd.DataFrame(inspection_rows)
display(global_inspection)

,dataset_id,n_archivos,shape,columnas_normalizadas
0,ser_autorizaciones,4,"(1170737, 17)","[tipo_autorizacion, subtipo_autorizacion, cod_..."
1,ser_padron_vehiculos_ivtm_barrio,3,"(617349, 17)","[ejercicio, cod_tipo_persona, tipo_persona, co..."


## 5. Limpieza de `ser_autorizaciones`

### Qué mide

Mide autorizaciones SER activas en algún momento del periodo: residentes y comerciales, mensuales o anuales.

### Uso en el TFM

Sirve para aproximar presión estructural potencial no observada directamente en los tiques pagados. La fuente no mide ocupación horaria ni vehículos únicos verificables; mide registros de autorización.

### Criterio metodológico

La unidad observable del archivo es la autorización registrada. Al no existir matrícula, identificador de vehículo ni identificador único de autorización, no se eliminan supuestos duplicados por coincidencia de atributos. Varias autorizaciones pueden compartir barrio, fechas, distintivo, periodicidad o subtipo.

### Columnas conservadas

Se conservan:

- `tipo_autorizacion`;
- `subtipo_autorizacion`;
- `cod_distrito`;
- `distrito`;
- `cod_distrito_barrio_origen`;
- `num_barrio`;
- `cod_barrio`;
- `barrio`;
- `ambito_espacial`;
- `estado`;
- `fecha_activacion`;
- `fecha_vigencia`;
- `periodicidad`.

### Columnas descartadas

- `fecha_inicio_periodo`: no se conserva porque no es homogénea en todos los años; se usa `fecha_activacion` como fecha operativa común.
- `distintivo`: no se conserva porque no existen autorizaciones con distintivo CERO.
- `tipologia_vehiculo`: se inspecciona, pero no se usa para filtrar ni se conserva.
- `tipologia_propulsion` y `clasificacion_industria`: aumentan granularidad sin aportar señal necesaria para la presión estructural inicial.

### Validaciones

Se comprueba que las fechas operativas son parseables, que los estados no operativos se excluyen, que los intervalos temporales son lógicos, que `CUALQUIERA` queda limitado a autorizaciones comercial-industriales, que los ámbitos compuestos SER se identifican explícitamente y que el código de barrio se armoniza al formato usado en el resto del bloque SER cuando existe correspondencia con un barrio administrativo único.


### 5.1. Normalización y checks mínimos

Este bloque prepara la fuente para limpieza. No se muestran tablas EDA extensas: solo checks que justifican decisiones de filtrado o transformación.


In [117]:
EXPECTED_AUTH_COLS_DOC = [
    "tipo_autorizacion",
    "subtipo_autorizacion",
    "cod_distrito",
    "distrito",
    "cod_distrito_barrio",
    "barrio",
    "estado",
    "fecha_inicio_periodo",
    "fecha_activacion",
    "fecha_vigencia",
    "distintivo",
    "tipologia_vehiculo",
    "tipologia_propulsion",
    "periodicidad",
    "clasificacion_industria",
]

AUTH_FINAL_COLS = [
    "tipo_autorizacion",
    "subtipo_autorizacion",
    "cod_distrito",
    "distrito",
    "cod_distrito_barrio_origen",
    "num_barrio",
    "cod_barrio",
    "barrio",
    "ambito_espacial",
    "estado",
    "fecha_activacion",
    "fecha_vigencia",
    "periodicidad",
]

AUTH_VALID_ESTADOS_PRESION = {"ACTIVO", "BAJA"}

ser_autorizaciones_raw = RAW_TABLES["ser_autorizaciones"].copy()

for col in EXPECTED_AUTH_COLS_DOC:
    if col not in ser_autorizaciones_raw.columns:
        ser_autorizaciones_raw[col] = pd.NA


def to_nullable_int_if_integer(series: pd.Series) -> pd.Series:
    num = to_numeric_series(series)
    non_null = num.dropna()
    if len(non_null) == 0:
        return num.astype("Int64")
    if ((non_null % 1) == 0).all():
        return num.astype("Int64")
    return num


auth = ser_autorizaciones_raw.copy()

text_cols = [
    "tipo_autorizacion",
    "subtipo_autorizacion",
    "distrito",
    "barrio",
    "estado",
    "distintivo",
    "tipologia_vehiculo",
    "tipologia_propulsion",
    "periodicidad",
]
for col in text_cols:
    auth[col] = auth[col].map(normalize_text).astype("string")

auth["cod_distrito"] = to_nullable_int_if_integer(auth["cod_distrito"])
auth["cod_distrito_barrio_raw"] = to_nullable_int_if_integer(auth["cod_distrito_barrio"])
auth["cod_distrito_barrio_origen"] = auth["cod_distrito_barrio_raw"]

for col in ["fecha_inicio_periodo", "fecha_activacion", "fecha_vigencia"]:
    auth[col] = parse_date_series(auth[col])

auth["tipo_autorizacion_norm"] = auth["tipo_autorizacion"].str.upper()
auth["estado_norm"] = auth["estado"].str.upper()
auth["barrio_norm"] = auth["barrio"].str.upper()
auth["distintivo_norm"] = auth["distintivo"].str.upper()
auth["tipologia_vehiculo_norm"] = auth["tipologia_vehiculo"].str.upper()

# Armonización territorial:
# La fuente puede traer cod_distrito_barrio en formato compacto (21, 101, 214)
# o en formato canónico (201, 1001, 2104). Se prueban ambos patrones y
# se reconstruye siempre cod_barrio = cod_distrito * 100 + num_barrio.
auth["num_barrio"] = pd.Series(pd.NA, index=auth.index, dtype="Int64")
mask_codigo_barrio = auth["cod_distrito"].notna() & auth["cod_distrito_barrio_raw"].notna()

candidate_compact = (
    auth.loc[mask_codigo_barrio, "cod_distrito_barrio_raw"]
    - auth.loc[mask_codigo_barrio, "cod_distrito"] * 10
)

candidate_canonical = (
    auth.loc[mask_codigo_barrio, "cod_distrito_barrio_raw"]
    - auth.loc[mask_codigo_barrio, "cod_distrito"] * 100
)

compact_valid = candidate_compact.between(1, 9)
canonical_valid = candidate_canonical.between(1, 9)

auth.loc[mask_codigo_barrio, "num_barrio"] = np.select(
    [compact_valid, canonical_valid],
    [candidate_compact, candidate_canonical],
    default=pd.NA,
)

auth["num_barrio"] = auth["num_barrio"].astype("Int64")

auth["cod_barrio"] = pd.Series(pd.NA, index=auth.index, dtype="Int64")
mask_num_barrio = auth["cod_distrito"].notna() & auth["num_barrio"].notna()

auth.loc[mask_num_barrio, "cod_barrio"] = (
    auth.loc[mask_num_barrio, "cod_distrito"] * 100
    + auth.loc[mask_num_barrio, "num_barrio"]
).astype("Int64")

auth["formato_cod_barrio_origen"] = pd.Series(pd.NA, index=auth.index, dtype="string")
auth.loc[mask_codigo_barrio[mask_codigo_barrio].index[compact_valid], "formato_cod_barrio_origen"] = "compacto"
auth.loc[mask_codigo_barrio[mask_codigo_barrio].index[~compact_valid & canonical_valid], "formato_cod_barrio_origen"] = "canonico"

available_doc_cols = [col for col in EXPECTED_AUTH_COLS_DOC if col in ser_autorizaciones_raw.columns]
missing_doc_cols = [col for col in EXPECTED_AUTH_COLS_DOC if col not in ser_autorizaciones_raw.columns]
extra_cols = [
    col for col in ser_autorizaciones_raw.columns
    if col not in EXPECTED_AUTH_COLS_DOC + ["archivo_origen", "anio_archivo"]
]

possible_id_cols = [
    col for col in extra_cols
    if any(token in col.lower() for token in [
        "matricula", "id", "ident", "referencia", "expediente",
        "num_autorizacion", "id_autorizacion", "id_vehiculo"
    ])
]

valid_status_mask = auth["estado_norm"].isin(AUTH_VALID_ESTADOS_PRESION)

valid_temporal_mask = (
    auth["fecha_activacion"].notna()
    & auth["fecha_vigencia"].notna()
    & auth["fecha_activacion"].le(auth["fecha_vigencia"])
    & auth["fecha_activacion"].le(WINDOW_END)
    & auth["fecha_vigencia"].ge(WINDOW_START)
)

barrio_is_cualquiera = auth["barrio_norm"].fillna("").eq("CUALQUIERA")
tipo_is_comercial = auth["tipo_autorizacion_norm"].fillna("").eq("VEHICULO COMERCIAL-INDUSTRIAL")

BARRIOS_COMPUESTOS_SER = {"SOL-PALACIO", "SOL-CORTES"}
barrio_is_compuesto_ser = auth["barrio_norm"].isin(BARRIOS_COMPUESTOS_SER)

auth["ambito_espacial"] = np.select(
    [
        barrio_is_cualquiera,
        barrio_is_compuesto_ser,
        auth["cod_barrio"].notna(),
    ],
    [
        "comercial_cualquiera",
        "barrio_compuesto_ser",
        "barrio",
    ],
    default="revisar",
)
auth["ambito_espacial"] = pd.Series(auth["ambito_espacial"], dtype="string")

estado_counts = auth["estado_norm"].value_counts(dropna=False).to_dict()
tipologia_top5 = auth["tipologia_vehiculo_norm"].value_counts(dropna=False).head(5).to_dict()

auth_checks = pd.DataFrame([
    {"check": "filas_raw", "valor": len(auth)},
    {"check": "anios_archivo", "valor": sorted(auth["anio_archivo"].dropna().unique().tolist())},
    {"check": "esquema_documental_ok", "valor": not missing_doc_cols and not extra_cols},
    {"check": "columnas_identificador", "valor": possible_id_cols if possible_id_cols else "no_disponible"},
    {"check": "fecha_activacion_parseo_nulo", "valor": int(auth["fecha_activacion"].isna().sum())},
    {"check": "fecha_vigencia_parseo_nulo", "valor": int(auth["fecha_vigencia"].isna().sum())},
    {"check": "fecha_activacion_gt_fecha_vigencia", "valor": int(auth["fecha_activacion"].gt(auth["fecha_vigencia"]).sum())},
    {"check": "estados_raw", "valor": estado_counts},
    {"check": "filas_estado_no_operativo", "valor": int((~valid_status_mask).sum())},
    {"check": "barrio_nulo", "valor": int(auth["barrio"].isna().sum())},
    {"check": "cod_barrio_nulo_revisar", "valor": int((auth["cod_barrio"].isna() & auth["ambito_espacial"].eq("revisar")).sum())},
    {"check": "cod_barrio_nulo_no_asignable", "valor": int((auth["cod_barrio"].isna() & auth["ambito_espacial"].isin(["comercial_cualquiera", "barrio_compuesto_ser"])).sum())},
    {"check": "cualquiera_operativo_no_comercial", "valor": int((valid_status_mask & barrio_is_cualquiera & ~tipo_is_comercial).sum())},
    {"check": "barrios_compuestos_ser", "valor": auth.loc[barrio_is_compuesto_ser, "barrio_norm"].value_counts(dropna=False).to_dict()},
    {"check": "formato_cod_barrio_origen", "valor": auth["formato_cod_barrio_origen"].value_counts(dropna=False).to_dict()},
    {"check": "autorizaciones_distintivo_cero", "valor": int(auth["distintivo_norm"].eq("CERO").sum())},
    {"check": "tipologia_vehiculo_top5", "valor": tipologia_top5},
])

display(auth_checks)


,check,valor
0,filas_raw,1170737
1,anios_archivo,"[2023, 2024, 2025, 2026]"
2,esquema_documental_ok,True
3,columnas_identificador,no_disponible
4,fecha_activacion_parseo_nulo,0
5,fecha_vigencia_parseo_nulo,0
6,fecha_activacion_gt_fecha_vigencia,19
7,estados_raw,"{'ACTIVO': 957695, 'BAJA': 211876, 'RECHAZADO'..."
8,filas_estado_no_operativo,1166
9,barrio_nulo,0


**Lectura de checks.**

La estructura coincide con la referencia documental y no hay columnas identificadoras útiles, por lo que no se deduplican autorizaciones por coincidencia de atributos. `fecha_activacion` y `fecha_vigencia` son las fechas operativas homogéneas; los registros con fecha de activación posterior a la vigencia se excluyen. Los estados no operativos quedan fuera.

La armonización territorial distingue tres casos: autorizaciones asignables a un barrio administrativo único, autorizaciones comercial-industriales con `barrio = CUALQUIERA` y autorizaciones con ámbito compuesto SER (`SOL-PALACIO`, `SOL-CORTES`). Estas últimas no se imputan a un barrio único porque producirían un join espacial arbitrario.


### 5.2. Construcción de `ser_autorizaciones_clean`

Reglas aplicadas:

1. conservar solo `ACTIVO` y `BAJA`;
2. conservar solo registros con intervalo temporal lógico e intersección con la ventana del TFM;
3. armonizar `cod_distrito_barrio` a `num_barrio` y `cod_barrio` cuando representa un barrio administrativo único;
4. conservar `CUALQUIERA` y ámbitos compuestos SER sin imputación territorial;
5. no deduplicar por atributos;
6. conservar solo columnas útiles para presión estructural.


In [118]:
auth_clean_candidate = auth.loc[valid_status_mask & valid_temporal_mask].copy()

ser_autorizaciones_clean = (
    auth_clean_candidate[AUTH_FINAL_COLS]
    .sort_values(["fecha_activacion", "fecha_vigencia", "cod_barrio", "tipo_autorizacion"], na_position="last")
    .reset_index(drop=True)
)

auth_clean_quality = pd.DataFrame([
    {"check": "filas_raw", "valor": len(auth)},
    {"check": "filas_excluidas_estado_no_operativo", "valor": int((~valid_status_mask).sum())},
    {"check": "filas_excluidas_fecha_invalida_o_fuera_ventana", "valor": int((valid_status_mask & ~valid_temporal_mask).sum())},
    {"check": "filas_clean", "valor": len(ser_autorizaciones_clean)},
    {"check": "barrio_nulo_clean", "valor": int(ser_autorizaciones_clean["barrio"].isna().sum())},
    {"check": "cod_barrio_nulo_clean", "valor": int(ser_autorizaciones_clean["cod_barrio"].isna().sum())},
    {"check": "cod_barrio_nulo_revisar_clean", "valor": int((ser_autorizaciones_clean["cod_barrio"].isna() & ser_autorizaciones_clean["ambito_espacial"].eq("revisar")).sum())},
    {"check": "ambito_espacial_clean", "valor": ser_autorizaciones_clean["ambito_espacial"].value_counts(dropna=False).to_dict()},
    {"check": "cualquiera_clean", "valor": int(ser_autorizaciones_clean["barrio"].str.upper().eq("CUALQUIERA").sum())},
    {"check": "barrios_compuestos_clean", "valor": ser_autorizaciones_clean.loc[ser_autorizaciones_clean["ambito_espacial"].eq("barrio_compuesto_ser"), "barrio"].value_counts(dropna=False).to_dict()},
])

display(auth_clean_quality)

print("Vista preliminar ser_autorizaciones_clean:")
display(ser_autorizaciones_clean.head())

print("Tipos de ser_autorizaciones_clean:")
display(
    ser_autorizaciones_clean
    .dtypes
    .astype(str)
    .reset_index()
    .rename(columns={"index": "columna", 0: "dtype"})
)


,check,valor
0,filas_raw,1170737
1,filas_excluidas_estado_no_operativo,1166
2,filas_excluidas_fecha_invalida_o_fuera_ventana,19
3,filas_clean,1169552
4,barrio_nulo_clean,0
5,cod_barrio_nulo_clean,135746
6,cod_barrio_nulo_revisar_clean,0
7,ambito_espacial_clean,"{'barrio': 1033806, 'comercial_cualquiera': 13..."
8,cualquiera_clean,133549
9,barrios_compuestos_clean,"{'SOL-PALACIO': 1609, 'SOL-CORTES': 588}"


Vista preliminar ser_autorizaciones_clean:


,tipo_autorizacion,subtipo_autorizacion,cod_distrito,distrito,cod_distrito_barrio_origen,num_barrio,cod_barrio,barrio,ambito_espacial,estado,fecha_activacion,fecha_vigencia,periodicidad
0,VEHICULO COMERCIAL-INDUSTRIAL,COMERCIAL 5h,<NA>,<NA>,<NA>,<NA>,<NA>,CUALQUIERA,comercial_cualquiera,BAJA,2022-09-29,2023-02-28,MENSUAL
1,RESIDENTE,Titular vehículo,2,ARGANZUELA,21,1,201,IMPERIAL,barrio,ACTIVO,2022-09-29,2023-12-31,ANUAL
2,RESIDENTE,Titular vehículo,2,ARGANZUELA,22,2,202,LAS ACACIAS,barrio,ACTIVO,2022-09-30,2023-12-31,ANUAL
3,RESIDENTE,Titular vehículo,2,ARGANZUELA,26,6,206,PALOS DE MOGUER,barrio,ACTIVO,2022-09-30,2023-12-31,ANUAL
4,RESIDENTE,Titular vehículo,3,RETIRO,31,1,301,PACÍFICO,barrio,ACTIVO,2022-09-30,2023-12-31,ANUAL


Tipos de ser_autorizaciones_clean:


,columna,dtype
0,tipo_autorizacion,string
1,subtipo_autorizacion,string
2,cod_distrito,Int64
3,distrito,string
4,cod_distrito_barrio_origen,Int64
5,num_barrio,Int64
6,cod_barrio,Int64
7,barrio,string
8,ambito_espacial,string
9,estado,string


**Lectura/resumen del bloque `ser_autorizaciones`.**

La fuente `ser_autorizaciones` queda preparada como tabla limpia de autorizaciones SER operativas dentro de la ventana temporal del TFM. Se conservan únicamente registros con estado `ACTIVO` o `BAJA`, y se excluyen los estados no operativos (`RECHAZADO`, `EN TRÁMITE` y `PENDIENTE ACTIVACION`) porque no representan autorizaciones efectivas para aproximar presión estructural. También se eliminan los registros con intervalo temporal ilógico, definidos como aquellos en los que `fecha_activacion` es posterior a `fecha_vigencia`.

La fecha operativa común de la fuente será `fecha_activacion`, ya que `fecha_inicio_periodo` no presenta cobertura homogénea en todos los años. La columna `distintivo` no se conserva porque no existen autorizaciones con distintivo CERO, por lo que no aporta señal útil para el cruce posterior con IVTM. Tampoco se conserva `tipologia_vehiculo`, ya que no existe una regla documental que permita excluir autorizaciones válidas por tipo de vehículo sin introducir un criterio arbitrario.

La armonización territorial distingue tres casos: autorizaciones asignables a un barrio administrativo único, autorizaciones comercial-industriales con `barrio = CUALQUIERA` y autorizaciones con ámbito compuesto SER (`SOL-PALACIO`, `SOL-CORTES`). Los dos últimos casos se conservan con `cod_barrio` nulo y con `ambito_espacial` explícito, porque no deben forzarse a un único barrio para joins posteriores. Esta decisión evita introducir errores espaciales en el panel SER.

## 6. Limpieza de `ser_padron_vehiculos_ivtm_barrio`

### Qué mide

Mide el padrón anual IVTM de vehículos del municipio de Madrid por ejercicio, distrito, barrio, tipo de vehículo, etiqueta medioambiental, clasificación ambiental, carburante, año de matriculación y contador.

### Uso en el TFM

Se transforma en una tabla anual por barrio que aproxima presión estructural potencial de turismos Cero Emisiones. Es una señal complementaria, no ocupación observada ni presencia horaria en calle.

### Unidad final

La unidad final es `barrio-año`. Solo se conservan registros con barrio administrativo real y `cod_barrio` canónico. Los registros sin barrio asignable se diagnostican, pero se excluyen del clean final porque no pueden enlazarse de forma fiable con tiques, barrios SER o paneles espaciales.

### Columnas conservadas

El clean final agregado conserva:

- `anio`;
- `cod_distrito`;
- `distrito`;
- `cod_barrio_origen`;
- `num_barrio`;
- `cod_barrio`;
- `barrio`;
- `n_turismos_distintivo_0`.

### Criterio metodológico

Aunque `ETIQUETA_MEDIOAMBIENTAL = 0` identifica vehículos Cero Emisiones, no todos los tipos de vehículo son comparables para medir presión estructural sobre plazas SER ordinarias. La inspección de la fuente muestra que la etiqueta `0` incluye turismos, motocicletas, ciclomotores, camiones, autobuses y tractores. Por ello, la variable limpia se restringe a turismos (`COD_TIPO_VEHICULO = "TU"`), que son la señal más coherente con el problema de aparcamiento regulado en superficie.

### Validaciones

Se comprueba que los años del archivo coinciden con `EJERCICIO`, que `CONTADOR` es numérico, entero y no negativo, que la etiqueta `0` existe, que la clasificación ambiental es coherente, que los códigos de barrio se armonizan al formato SER y que la agregación conserva la suma de turismos CERO antes y después del agrupamiento. Los registros sin barrio se cuantifican como exclusión metodológica, no como error.


### 6.1. Normalización territorial, ambiental y de conteos

Este bloque normaliza la fuente y concentra las comprobaciones necesarias. La tabla de composición de la etiqueta `0` por tipo de vehículo se mantiene como evidencia metodológica: permite justificar por qué el clean final se restringe a turismos Cero Emisiones.


In [119]:
EXPECTED_IVTM_COLS = [
    "ejercicio",
    "cod_tipo_persona",
    "tipo_persona",
    "cod_distrito",
    "distrito",
    "cod_barrio",
    "barrio",
    "cod_tipo_vehiculo",
    "tipo_vehiculo",
    "etiqueta_medioambiental",
    "clasificacion_ambiental",
    "cuota",
    "tipo_carburante",
    "ano_matriculacion",
    "contador",
]

IVTM_FINAL_COLS = [
    "anio",
    "cod_distrito",
    "distrito",
    "cod_barrio_origen",
    "num_barrio",
    "cod_barrio",
    "barrio",
    "ambito_espacial",
    "n_turismos_distintivo_0",
]

ivtm_raw = RAW_TABLES["ser_padron_vehiculos_ivtm_barrio"].copy()

for col in EXPECTED_IVTM_COLS:
    if col not in ivtm_raw.columns:
        ivtm_raw[col] = pd.NA

ivtm = ivtm_raw.copy()

text_cols = [
    "cod_tipo_persona",
    "tipo_persona",
    "distrito",
    "barrio",
    "cod_tipo_vehiculo",
    "tipo_vehiculo",
    "etiqueta_medioambiental",
    "clasificacion_ambiental",
    "tipo_carburante",
]

for col in text_cols:
    if col == "etiqueta_medioambiental":
        # En esta fuente, etiqueta vacía no equivale a NA operativo:
        # representa "Sin distintivo Ambiental" según la clasificación documental.
        ivtm[col] = ivtm[col].astype("string").fillna("").map(normalize_text).astype("string")
    else:
        ivtm[col] = ivtm[col].map(normalize_text).astype("string")

ivtm["anio"] = to_nullable_int_if_integer(ivtm["ejercicio"])
ivtm["cod_distrito"] = to_nullable_int_if_integer(ivtm["cod_distrito"])
ivtm["cod_barrio_origen"] = to_nullable_int_if_integer(ivtm["cod_barrio"])
ivtm["ano_matriculacion"] = to_nullable_int_if_integer(ivtm["ano_matriculacion"])
ivtm["contador_num"] = to_numeric_series(ivtm["contador"])

ivtm["etiqueta_norm"] = (
    ivtm["etiqueta_medioambiental"]
    .astype("string")
    .fillna("")
    .str.strip()
    .str.upper()
    .replace({"E": "ECO", "0.0": "0"})
)

ivtm["clasificacion_norm"] = (
    ivtm["clasificacion_ambiental"]
    .astype("string")
    .fillna("")
    .str.strip()
    .str.upper()
)

ivtm["cod_tipo_vehiculo_norm"] = (
    ivtm["cod_tipo_vehiculo"]
    .astype("string")
    .fillna("")
    .str.strip()
    .str.upper()
)

ivtm["barrio_norm"] = (
    ivtm["barrio"]
    .astype("string")
    .fillna("")
    .str.strip()
    .str.upper()
)

# Armonización territorial.
# La fuente usa cod_barrio de origen en formato compacto por distrito:
# Centro/Palacio aparece como 11, no como 101. Se conserva el código original
# y se crea cod_barrio canónico = cod_distrito * 100 + num_barrio cuando existe barrio real.
ivtm["num_barrio"] = pd.Series(pd.NA, index=ivtm.index, dtype="Int64")
mask_cod_barrio_origen = ivtm["cod_distrito"].notna() & ivtm["cod_barrio_origen"].notna()

candidate_compact = (
    ivtm.loc[mask_cod_barrio_origen, "cod_barrio_origen"]
    - ivtm.loc[mask_cod_barrio_origen, "cod_distrito"] * 10
)

candidate_canonical = (
    ivtm.loc[mask_cod_barrio_origen, "cod_barrio_origen"]
    - ivtm.loc[mask_cod_barrio_origen, "cod_distrito"] * 100
)

compact_valid = candidate_compact.between(0, 9)
canonical_valid = candidate_canonical.between(0, 9)

ivtm.loc[mask_cod_barrio_origen, "num_barrio"] = np.select(
    [compact_valid, canonical_valid],
    [candidate_compact, candidate_canonical],
    default=pd.NA,
)
ivtm["num_barrio"] = ivtm["num_barrio"].astype("Int64")

ivtm["cod_barrio"] = pd.Series(pd.NA, index=ivtm.index, dtype="Int64")

mask_barrio_real = (
    ivtm["cod_distrito"].notna()
    & ivtm["cod_distrito"].ne(0)
    & ivtm["num_barrio"].between(1, 9)
    & ~ivtm["barrio_norm"].eq("--")
)

ivtm.loc[mask_barrio_real, "cod_barrio"] = (
    ivtm.loc[mask_barrio_real, "cod_distrito"] * 100
    + ivtm.loc[mask_barrio_real, "num_barrio"]
).astype("Int64")

ivtm["ambito_espacial"] = np.select(
    [
        mask_barrio_real,
        ivtm["cod_distrito"].eq(0) | ivtm["distrito"].astype("string").str.strip().eq("--"),
        ivtm["cod_distrito"].gt(0) & ivtm["barrio_norm"].eq("--"),
    ],
    [
        "barrio",
        "municipio_sin_distrito",
        "distrito_sin_barrio",
    ],
    default="revisar",
)
ivtm["ambito_espacial"] = pd.Series(ivtm["ambito_espacial"], dtype="string")

ivtm["is_distintivo_0"] = ivtm["etiqueta_norm"].eq("0")
ivtm["is_turismo"] = ivtm["cod_tipo_vehiculo_norm"].eq("TU")

ivtm["contador_turismo_cero"] = np.where(
    ivtm["is_distintivo_0"] & ivtm["is_turismo"],
    ivtm["contador_num"],
    0,
)

# Composición de lo que se está contando como etiqueta 0.
ivtm_zero_vehicle_type_check = (
    ivtm[ivtm["is_distintivo_0"]]
    .groupby(["cod_tipo_vehiculo_norm", "tipo_vehiculo"], dropna=False)
    .agg(
        n_filas=("contador_num", "size"),
        n_vehiculos_distintivo_0=("contador_num", "sum"),
    )
    .reset_index()
    .sort_values("n_vehiculos_distintivo_0", ascending=False)
    .reset_index(drop=True)
)

ivtm_zero_vehicle_type_check["n_vehiculos_distintivo_0"] = (
    ivtm_zero_vehicle_type_check["n_vehiculos_distintivo_0"]
    .round(0)
    .astype("Int64")
)

display(ivtm_zero_vehicle_type_check)

# Checks compactos.
archivo_ejercicio_mismatch = (
    ivtm[ivtm["anio_archivo"].notna() & ivtm["anio"].notna()]
    .assign(match=lambda df: df["anio_archivo"].astype("Int64").eq(df["anio"].astype("Int64")))
    .loc[lambda df: ~df["match"]]
    .shape[0]
)

contador_non_null = ivtm["contador_num"].dropna()
contador_no_entero = int(((contador_non_null % 1) != 0).sum())
contador_negativo = int((contador_non_null < 0).sum())

etiquetas_presentes = sorted([str(x) for x in ivtm["etiqueta_norm"].dropna().unique().tolist()])
anios_con_etiqueta_0 = set(ivtm.loc[ivtm["is_distintivo_0"], "anio"].dropna().astype(int).unique().tolist())

cero_clasificacion_no_cero = int(
    (
        ivtm["is_distintivo_0"]
        & ~ivtm["clasificacion_norm"].str.contains("CERO", na=False)
    ).sum()
)

empty_label_sin_distintivo_incoherente = int(
    (
        ivtm["etiqueta_norm"].eq("")
        & ~ivtm["clasificacion_norm"].str.contains("SIN DISTINTIVO", na=False)
    ).sum()
)

dash_label_sin_clasificacion_incoherente = int(
    (
        ivtm["etiqueta_norm"].eq("--")
        & ~ivtm["clasificacion_norm"].str.contains("SIN CLASIFIC", na=False)
    ).sum()
)

sum_turismo_cero_antes = int(ivtm["contador_turismo_cero"].sum())

ivtm_checks = pd.DataFrame([
    {"check": "filas_raw", "valor": len(ivtm)},
    {"check": "anios_archivo", "valor": sorted(ivtm["anio_archivo"].dropna().unique().tolist())},
    {"check": "ejercicios", "valor": sorted(ivtm["anio"].dropna().unique().tolist())},
    {"check": "anios_esperados_ivtm", "valor": sorted(EXPECTED_YEARS_IVTM)},
    {"check": "archivo_ejercicio_mismatch", "valor": int(archivo_ejercicio_mismatch)},
    {"check": "contador_parseo_nulo", "valor": int(ivtm["contador_num"].isna().sum())},
    {"check": "contador_no_entero", "valor": contador_no_entero},
    {"check": "contador_negativo", "valor": contador_negativo},
    {"check": "etiquetas_ambientales_presentes", "valor": etiquetas_presentes},
    {"check": "etiqueta_0_presente_todos_anios", "valor": EXPECTED_YEARS_IVTM.issubset(anios_con_etiqueta_0)},
    {"check": "n_turismos_distintivo_0_total", "valor": sum_turismo_cero_antes},
    {"check": "etiqueta_0_clasificacion_no_cero", "valor": cero_clasificacion_no_cero},
    {"check": "etiqueta_vacia_no_sin_distintivo", "valor": empty_label_sin_distintivo_incoherente},
    {"check": "etiqueta_--_no_sin_clasificacion", "valor": dash_label_sin_clasificacion_incoherente},
    {"check": "ambito_espacial_raw", "valor": ivtm["ambito_espacial"].value_counts(dropna=False).to_dict()},
    {"check": "cod_barrio_canónico_revisar", "valor": int(ivtm["ambito_espacial"].eq("revisar").sum())},
    {"check": "suma_turismos_cero_antes_agregar", "valor": sum_turismo_cero_antes},
])

display(ivtm_checks)


,cod_tipo_vehiculo_norm,tipo_vehiculo,n_filas,n_vehiculos_distintivo_0
0,TU,TURISMO,23265,94755
1,MT,MOTOCICLETA,3161,33335
2,CI,CICLOMOTOR,2189,10270
3,CA,CAMION,1470,5614
4,AU,AUTOBUS,82,1127
5,TR,TRACTOR,36,153


,check,valor
0,filas_raw,617349
1,anios_archivo,"[2023, 2024, 2025]"
2,ejercicios,"[2023, 2024, 2025]"
3,anios_esperados_ivtm,"[2023, 2024, 2025]"
4,archivo_ejercicio_mismatch,0
5,contador_parseo_nulo,0
6,contador_no_entero,0
7,contador_negativo,0
8,etiquetas_ambientales_presentes,"[, --, 0, B, C, ECO]"
9,etiqueta_0_presente_todos_anios,True


**Lectura de checks.**

La tabla de composición de etiqueta `0` muestra que los vehículos Cero Emisiones no son únicamente turismos: también aparecen motocicletas, ciclomotores, camiones, autobuses y tractores. Por tanto, el clean final se restringe a turismos Cero Emisiones (`COD_TIPO_VEHICULO = "TU"`), al ser la señal más coherente con la presión estructural sobre plazas SER ordinarias.

Los registros sin barrio administrativo real se clasifican como `municipio_sin_distrito` o `distrito_sin_barrio`. No se imputan mediante join porque no contienen un barrio textual verificable; forzarlos a un barrio introduciría error espacial. Los registros con `ambito_espacial = revisar` deben ser cero antes de cerrar la fuente.


### 6.2. Construcción de `ser_padron_vehiculos_ivtm_barrio_clean`

Primero se agrega por `anio`, distrito y barrio, conservando la clasificación espacial (`barrio`, `municipio_sin_distrito`, `distrito_sin_barrio`). Después, el clean final se restringe a `ambito_espacial = "barrio"`.

La salida conserva únicamente `n_turismos_distintivo_0`, definida como la suma de `CONTADOR` para registros con `ETIQUETA_MEDIOAMBIENTAL = "0"` y `COD_TIPO_VEHICULO = "TU"`. No se conservan motocicletas, ciclomotores, camiones, autobuses ni tractores Cero Emisiones porque no representan la misma presión sobre plazas SER ordinarias que los turismos.


In [122]:
ivtm_group_cols = [
    "anio",
    "cod_distrito",
    "distrito",
    "cod_barrio_origen",
    "num_barrio",
    "cod_barrio",
    "barrio",
    "ambito_espacial",
]

ivtm_agg_all = (
    ivtm
    .groupby(ivtm_group_cols, dropna=False)
    .agg(
        n_turismos_distintivo_0=("contador_turismo_cero", "sum"),
    )
    .reset_index()
    .sort_values(["anio", "cod_distrito", "cod_barrio"], na_position="last")
    .reset_index(drop=True)
)

ivtm_agg_all["n_turismos_distintivo_0"] = (
    ivtm_agg_all["n_turismos_distintivo_0"]
    .round(0)
    .astype("Int64")
)

turismos_cero_agg_total = int(ivtm_agg_all["n_turismos_distintivo_0"].sum())
turismos_cero_excluidos_no_barrio = int(
    ivtm_agg_all.loc[
        ~ivtm_agg_all["ambito_espacial"].eq("barrio"),
        "n_turismos_distintivo_0"
    ].sum()
)

ser_padron_vehiculos_ivtm_barrio_clean = (
    ivtm_agg_all
    .loc[ivtm_agg_all["ambito_espacial"].eq("barrio")]
    .drop(columns=["ambito_espacial"])
    .reset_index(drop=True)
)

sum_turismo_cero_despues = int(
    ser_padron_vehiculos_ivtm_barrio_clean["n_turismos_distintivo_0"].sum()
)

ivtm_clean_quality = pd.DataFrame([
    {"check": "filas_raw", "valor": len(ivtm)},
    {"check": "filas_agg_total", "valor": len(ivtm_agg_all)},
    {"check": "filas_clean_barrio", "valor": len(ser_padron_vehiculos_ivtm_barrio_clean)},
    {"check": "ambito_espacial_agg_total", "valor": ivtm_agg_all["ambito_espacial"].value_counts(dropna=False).to_dict()},
    {"check": "cod_barrio_revisar_agg_total", "valor": int(ivtm_agg_all["ambito_espacial"].eq("revisar").sum())},
    {"check": "cod_barrio_nulo_clean", "valor": int(ser_padron_vehiculos_ivtm_barrio_clean["cod_barrio"].isna().sum())},
    {"check": "suma_turismos_cero_antes_agregar", "valor": sum_turismo_cero_antes},
    {"check": "suma_turismos_cero_agg_total", "valor": turismos_cero_agg_total},
    {"check": "turismos_cero_excluidos_no_barrio", "valor": turismos_cero_excluidos_no_barrio},
    {"check": "suma_turismos_cero_clean_barrio", "valor": sum_turismo_cero_despues},
    {"check": "conservacion_masa_total", "valor": sum_turismo_cero_antes == turismos_cero_agg_total},
    {"check": "clean_mas_excluidos_igual_total", "valor": (sum_turismo_cero_despues + turismos_cero_excluidos_no_barrio) == turismos_cero_agg_total},
])

display(ivtm_clean_quality)

print("Vista preliminar ser_padron_vehiculos_ivtm_barrio_clean:")
display(ser_padron_vehiculos_ivtm_barrio_clean.head())

print("Tipos de ser_padron_vehiculos_ivtm_barrio_clean:")
display(
    ser_padron_vehiculos_ivtm_barrio_clean
    .dtypes
    .astype(str)
    .reset_index()
    .rename(columns={"index": "columna", 0: "dtype"})
)


,check,valor
0,filas_raw,617349
1,filas_agg_total,459
2,filas_clean_barrio,393
3,ambito_espacial_agg_total,"{'barrio': 393, 'distrito_sin_barrio': 63, 'mu..."
4,cod_barrio_revisar_agg_total,0
5,cod_barrio_nulo_clean,0
6,suma_turismos_cero_antes_agregar,94755
7,suma_turismos_cero_agg_total,94755
8,turismos_cero_excluidos_no_barrio,17410
9,suma_turismos_cero_clean_barrio,77345


Vista preliminar ser_padron_vehiculos_ivtm_barrio_clean:


,anio,cod_distrito,distrito,cod_barrio_origen,num_barrio,cod_barrio,barrio,n_turismos_distintivo_0
0,2023,1,Centro,11,1,101,PALACIO,98
1,2023,1,Centro,12,2,102,EMBAJADORES,75
2,2023,1,Centro,13,3,103,CORTES,72
3,2023,1,Centro,14,4,104,JUSTICIA,123
4,2023,1,Centro,15,5,105,UNIVERSIDAD,121


Tipos de ser_padron_vehiculos_ivtm_barrio_clean:


,columna,dtype
0,anio,Int64
1,cod_distrito,Int64
2,distrito,string
3,cod_barrio_origen,Int64
4,num_barrio,Int64
5,cod_barrio,Int64
6,barrio,string
7,n_turismos_distintivo_0,Int64


**Lectura/decisión.**

La fuente `ser_padron_vehiculos_ivtm_barrio` queda agregada a nivel `barrio-año` y restringida a registros con barrio administrativo real. La salida conserva únicamente `n_turismos_distintivo_0`, calculada como suma de `CONTADOR` en registros con etiqueta Cero Emisiones y tipo de vehículo turismo.

Esta decisión evita mezclar la presión estructural asociada al aparcamiento ordinario con otros tipos de vehículo Cero Emisiones, como motocicletas, ciclomotores, camiones, autobuses o tractores. También evita incorporar registros sin barrio asignable, que no pueden enlazarse de forma fiable con tiques ni con capas espaciales por barrio. La agregación conserva la suma total antes del filtrado y cuantifica explícitamente los turismos CERO excluidos por falta de barrio.


## 8. Escritura de salidas limpias

Se escribirán únicamente las salidas `interim` de las dos fuentes limpias. No se escriben diagnósticos CSV, outputs `processed`, paneles ni proxy.


In [121]:
clean_outputs = {
    "ser_autorizaciones": ser_autorizaciones_clean,
    "ser_padron_vehiculos_ivtm_barrio": ser_padron_vehiculos_ivtm_barrio_clean,
}

write_rows = []

for dataset_id, df in clean_outputs.items():
    out_rel = target_catalog.loc[
        target_catalog["dataset_id"].eq(dataset_id),
        "archivo_interim"
    ].iloc[0]
    out_path = ROOT / out_rel
    out_path.parent.mkdir(parents=True, exist_ok=True)

    df.to_parquet(out_path, index=False)

    write_rows.append({
        "dataset_id": dataset_id,
        "archivo_interim": out_rel,
        "shape_escrita": df.shape,
        "size_mb": round(out_path.stat().st_size / 1024**2, 3),
        "estado_escritura": "OK",
    })

write_check = pd.DataFrame(write_rows)
display(write_check)


,dataset_id,archivo_interim,shape_escrita,size_mb,estado_escritura
0,ser_autorizaciones,data/interim/ser/ser_autorizaciones/ser_autori...,"(1169552, 13)",1.579,OK
1,ser_padron_vehiculos_ivtm_barrio,data/interim/ser/ser_padron_vehiculos_ivtm_bar...,"(393, 8)",0.011,OK


## 9. Cierre

Este notebook deja preparadas dos fuentes complementarias de presión estructural SER en formato `interim`:

- autorizaciones SER limpias y temporalmente validadas;
- stock anual de turismos CERO por barrio derivado del IVTM.

La salida IVTM se restringe a barrios administrativos reales porque su uso posterior requiere joins por `cod_barrio`. Los registros sin barrio asignable se diagnostican, pero no se conservan en el clean final.